## AAL calculation

#### The code is modified to calculate AAL for Turkey and Brush Creek probabilistic modeling

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from osgeo import gdal
import os

In [2]:
def getTifData(tif_path: str):
    """Read a WSE raster and return the gdal objects"""
    src = gdal.Open(tif_path)
    rb = src.GetRasterBand(1)
    gt = src.GetGeoTransform()
    proj = src.GetProjection()
    return rb, gt, src

In [3]:
def query(x: float, y: float, gt: any, rb: any) -> float:
    """Queries one specific cell in the rasterband given an x, y in the 
    geotransform
    """
    px = int((x-gt[0]) / gt[1])   
    py = int((y-gt[3]) / gt[5])   
    return rb.ReadAsArray(px,py,1,1)[0][0]

In [4]:
def StructureData(gdf_path, gt, rb):
    """Read structures, compute WSE values, and depth values"""

    gdf = gpd.read_file(gdf_path) 
    # Add a new column for WSE values
    gdf['wse'] = None
    
    for i, idx in enumerate(gdf.index):
        x = gdf.loc[idx, 'x']
        y = gdf.loc[idx, 'y']
        pixel_value = query(x, y, gt, rb)
        
        # Assign WSE value to the GeoDataFrame, if value is -9999 it assigns the same -9999 value
        gdf.at[idx, 'wse'] = -9999 if pixel_value == -9999 else pixel_value

        # Calculating actual depth of water based on wse, ground elevation and foundation height. If the wse value is -9999, the depth is also assigned as -9999
        gdf.at[idx,'depth'] = gdf.at[idx,'wse'] - gdf.at[idx,'ground_elv'] - gdf.at[idx,'found_ht'] if gdf.at[idx,'wse'] != -9999 else -9999
    return gdf

In [5]:
def new_occtype(row):
    """Reads the  occupancy type and foundation type for RES3 occupancy types and reclassifies based on whether a structure has basement or not.
    Since RES3A to RES3F needed to be reclassified on their availability of the basement.
    """
    if row['occtype'] in occtype_val and row['found_type'] == found_type_val:
        return f"{row['occtype']}-{row['found_type']}"
    elif row['occtype'] in occtype_val and row['found_type'] != found_type_val:
        return f"{row['occtype']}-NB"
    else:
        return row['occtype']

In [6]:
def wseBuildingPts(tif_path, gdf_path):
    """Assigns WSE values to each building points"""
    rb, gt, src = getTifData(tif_path)
    wse_result = StructureData(gdf_path, gt, rb)
    return wse_result

In [7]:
def damage_pct(gdf, damage_df):
    """Reads building data (gdf) and depth damage function data (damage_ddf) and computes damage percentage values based on the 
    depth of water for each building points. This function is adjusted to work for Turkey and Brush Creek datasets.
    """
    gdf['dmg_pct'] = None
    
    # Loop through each row in the GeoDataFrame
    for idx, row in gdf.iterrows():
        occupancy_type = row['occ_new']  # Adjust column name as needed
        depth = row['depth']  # Adjust column name as needed

        # Check if depth is -9999, assign 0 damage and continue
        # This is done, because if a building point has a depth value of -9999, then there is no water in the waster surface raster. So, assigning zero damage
        if depth == -9999:
            gdf.at[idx, 'dmg_pct'] = 0
            continue
    
        # Get the matching row from the damage DataFrame
        damage_row = damage_df[damage_df['OccuNSI'] == occupancy_type]

        # Extract depth and damage values from the damage DataFrame
        depth_values = np.array([
        float(str(col)[1:]) * (-1 if str(col).startswith('m') else 1) if str(col) != '0' else 0
        for col in damage_row.columns[5:]
        ])
    
        damage_values = damage_row.iloc[0, 5:].values.astype(float)
    
        # Interpolate damage percentage for the given depth
        interpolated_damage = np.interp(depth, depth_values, damage_values)
    
        # Assign the interpolated damage percentage value to the GeoDataFrame
        gdf.at[idx, 'dmg_pct'] = interpolated_damage

    return gdf

#### The code below utilizes the functions above and calculates WSE and depth values for each building based on the WSE raster. After WSE and depth calculation, it leverages the Depth Damage Function (DDF) from FEMA-HAZUS and calculates the correspoding damage percentage values based on those values for each building points

In [8]:
tif_file = r"C:\Users\dneupane\Documents\Probabilistic\script_AAL\WSE_Turkey_3000_L_q1_project.tif"
bld_file = r"C:\Users\dneupane\Documents\Probabilistic\script_AAL\shp_nsi\turkey_nsi.shp"
output_path = r"C:\Users\dneupane\Documents\Probabilistic\script_AAL\shp_nsi\turkey_nsi_wse.shp"

# calculates the WSE and depth, and adds two columns containing wse and depth in the building shapefile.
wse_depth = wseBuildingPts(tif_file, bld_file)

# Reclassifying RES3 occupancy type based on whether a structure has a basement or not
occtype_val = ["RES3A", "RES3B", "RES3C", "RES3D", "RES3E", "RES3F"]
found_type_val = "B" 
# Apply the function "new_occtype" to each row
wse_depth['occ_new'] = wse_depth.apply(new_occtype, axis=1)

# Calculate damage percentage value for each building based on DDF function and Building Geodataframe
ddf = pd.read_excel(r"C:\Users\dneupane\Documents\Probabilistic\script_AAL\ddf_final.xlsx")   #Depth damage function excel file
dmg_val = damage_pct(wse_depth, ddf)   #Calculation of damage percentage and adds a damage percentage column with damage pct values

In [10]:
dmg_val

,fd_id,bid,cbfips,st_damcat,occtype,bldgtype,num_story,sqft,found_type,found_ht,...,x,y,firmzone,grnd_elv_m,ground_elv,geometry,wse,depth,occ_new,dmg_pct
0,521112345,86C7X7RG+3MF-9-3-9-3,200910524181002,RES,RES3A,W,1.0,1972.27386,C,1.5,...,-94.723322,38.990196,None,320.284882,1050.803451,POINT (2243484.365 256080.233),1050.694336,-1.609115,RES3A-NB,0.0
1,521229257,86C7X7RR+MG3-3-4-3-3,200910524183005,RES,RES1-1SNB,W,1.0,1089.51000,C,1.5,...,-94.708696,38.991640,None,315.300812,1034.451515,POINT (2247619.867 256756.400),-9999,-9999.000000,RES1-1SNB,0
2,521618589,86F727WR+PW6-0-0-0-0,202090436001029,RES,RES1-2SNB,W,2.0,1168.00000,C,1.5,...,-94.707741,39.046799,None,309.218536,1014.496543,POINT (2247160.864 276842.480),-9999,-9999.000000,RES1-2SNB,0
3,521028125,86F727FG+642-2-2-2-3,200910523053024,RES,RES3C,W,4.0,5458.48000,B,0.5,...,-94.724735,39.023004,None,311.873657,1023.207570,POINT (2242650.926 268006.740),-9999,-9999.000000,RES3C-B,0
4,521027589,86C7X78G+R6V-13-17-13-14,200910524172006,COM,COM1,S,1.0,33497.20000,S,0.5,...,-94.724446,38.967107,None,323.549866,1061.515341,POINT (2243469.202 247664.888),-9999,-9999.000000,COM1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27679,521618499,86F728VV+W45-0-0-0-0,202090434002044,IND,IND2,C,2.0,1180.00000,S,0.5,...,-94.657150,39.044750,None,257.278625,844.090006,POINT (2261546.918 276623.170),857.72876,13.138754,IND2,48.277508
27680,521618535,86F7372X+89G-0-0-0-0,202090436001018,RES,RES1-2SNB,W,2.0,2336.00000,S,0.5,...,-94.701561,39.050816,None,312.806152,1026.266937,POINT (2248861.547 278368.363),-9999,-9999.000000,RES1-2SNB,0
27681,521618604,86F7383C+QCC-0-0-0-0,202090435001011,COM,COM3,S,2.0,900.00000,S,0.5,...,-94.678970,39.054440,None,297.433807,975.832733,POINT (2255224.445 279921.802),-9999,-9999.000000,COM3,0
27682,521618605,86F7383C+FC6-0-0-0-0,202090435001011,PUB,REL1,W,2.0,900.00000,S,0.5,...,-94.678993,39.053669,None,293.615204,963.304505,POINT (2255228.240 279641.102),964.326599,0.522094,REL1,5.220937


#### Note: 
Currently, both AAL and AEP are calculated for individual WSE raster. After the generation of outputs for all the events, it will be looped to calculate the eventual AAL and AEP values using all the output WSE rasters. This code is to check the logic behind the calculation.

## AEP raster calculation

In [11]:
def calculateAEP(tif_path, csv_path): #output_path):
    """Compute AEP by assigning probability weights based on raster names and computes AEP raster"""
    df = pd.read_excel(excel_file)  # Load probability weights
    df.set_index('wse', inplace=True) 

    aep_raster = None

    rb, gt, src = getTifData(tif_path)  # Get raster data
    wse = rb.ReadAsArray()

    # for first iteration
    if aep_raster is None:
        aep_raster = np.zeros_like(wse, dtype=np.float32)

    # Event probability weight from CSV file
    weight = df.loc[raster_name, 'weights']

    # Applying the event probability weight where WSE value exists
    mask = wse > 0  # Masking to only get non-zero WSE values
    aep_raster[mask] += wse[mask] * weight

    return aep_raster

In [ ]:
aep_val = calculateAEP(tif_file, weights_csv)